PROMPT:
Build an SVM models to predict the label of an image from a given image on the yelp Photos dataset


## Team Members and Tasks:

| Team Member | Task                                                                                                                                                                                                                                                     |
|--------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Mayisha Nazar      | I imported the required libraries and defined file paths for the JSON metadata and image directory. Then, I loaded the JSON file, selected a subset of images, constructed full image paths, and verified their existence. After cleaning up unnecessary columns, I defined an image preprocessing function, processed and resized images, and handled any missing or failed cases. I selected a random subset of images, ensuring sufficient data for sampling. Finally, I converted image arrays into a feature matrix, encoded labels, and split the data into training and testing sets while maintaining class balance. The dataset is now prepared for model training. |
| Thirth Patel     | I trained the SVM model using the Support Vector Classifier (SVC) from Scikit-learn, then predicted the label for a test image. I created a function to preprocess the image, make predictions, and handle errors. I evaluated the model by generating and plotting the confusion matrix, printing the classification report, and computing precision, recall, and F1-score. Lastly, I binarized the labels for multi-class AUC, computed the ROC curve and AUC for each class, and plotted the multi-class ROC curve to assess model performance. |


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

# scikit-learn for preprocessing, evaluation, and SVM
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.svm import SVC



In [5]:
import pandas as pd
import numpy as np
import os
from PIL import Image

# Define paths
json_path = r"D:\College\Term3\computervision\Yelp-Photos\Yelp Photos\yelp_photos\photos.json"
image_dir = r"D:\College\Term3\computervision\Yelp-Photos\Yelp Photos\yelp_photos\photos"

# Load the JSON file
# Load the JSON file
df = pd.read_json(json_path, lines=True)

# Select only 20,000 random images (modify this number as needed)
df = df.sample(n=10000, random_state=42)

# Create full image paths using the correct folder
df['image_path'] = df['photo_id'].apply(lambda x: os.path.join(image_dir, f"{x}.jpg"))

# Check if files exist
df['exists'] = df['image_path'].apply(os.path.exists)

# Show missing files
missing_files = df[~df['exists']]
if not missing_files.empty:
    print("\n⚠️ Warning: Some image files are missing! Here are the first 5 missing files:")
    display(missing_files[['photo_id', 'image_path']].head())
else:
    print("\n✅ All image paths are valid!")

# Drop the 'exists' column (not needed anymore)
df.drop(columns=['exists'], inplace=True)


✅ All image paths are valid!


In [6]:
# Define preprocessing function
IMG_SIZE = (32, 32)  # Resize to 32x32 to reduce memory usage

def preprocess_image(image_path):
    """Loads an image, converts to RGB, resizes to (32x32), and flattens it."""
    try:
        img = Image.open(image_path).convert('RGB')  # Ensure RGB mode
        img = img.resize(IMG_SIZE)  # Resize
        return np.array(img).flatten().astype(np.float16)  # Convert to float16 to save memory
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None  # Skip failed images

# Apply preprocessing
df['image_array'] = df['image_path'].apply(preprocess_image)

# Drop rows where images could not be processed
df.dropna(subset=['image_array'], inplace=True)

# Print results
print("\n✅ Preprocessing completed successfully!")
print(f"Total valid images processed: {len(df)}")

# Display first 5 processed images (flattened arrays)
df[['photo_id', 'image_array']].head()

❌ Error processing image D:\College\Term3\computervision\Yelp-Photos\Yelp Photos\yelp_photos\photos\1MOGQBWogR8oJr1WgERi9g.jpg: cannot identify image file 'D:\\College\\Term3\\computervision\\Yelp-Photos\\Yelp Photos\\yelp_photos\\photos\\1MOGQBWogR8oJr1WgERi9g.jpg'
❌ Error processing image D:\College\Term3\computervision\Yelp-Photos\Yelp Photos\yelp_photos\photos\pW1IPuTdLIUB61goirbXaA.jpg: cannot identify image file 'D:\\College\\Term3\\computervision\\Yelp-Photos\\Yelp Photos\\yelp_photos\\photos\\pW1IPuTdLIUB61goirbXaA.jpg'

✅ Preprocessing completed successfully!
Total valid images processed: 9998


,photo_id,image_array
32568,k_PSngRS22mSA1MypwrjPg,"[154.0, 121.0, 86.0, 173.0, 140.0, 101.0, 180...."
174911,D_94KivwVgitkzFIgE_KcQ,"[218.0, 223.0, 226.0, 222.0, 226.0, 228.0, 220..."
132444,Hf39P7_G_eRCqfVwvMDV6g,"[122.0, 112.0, 100.0, 124.0, 114.0, 102.0, 116..."
46744,agxl4sABeRXwjLL506KMrQ,"[159.0, 152.0, 134.0, 160.0, 153.0, 134.0, 161..."
85073,7cZ0MREN2TwAAX4nnirQlA,"[8.0, 4.0, 4.0, 9.0, 5.0, 4.0, 9.0, 5.0, 4.0, ..."


In [7]:
# Define the number of images to use
NUM_IMAGES = 10000  

# Ensure dataset has enough samples before sampling
if len(df) >= NUM_IMAGES:
    df_sample = df.sample(n=NUM_IMAGES, random_state=42)  # Random subset
else:
    df_sample = df  # Use all available data if less than NUM_IMAGES

# Print info
print(f"✅ Selected {len(df_sample)} images out of {len(df)} total.")


✅ Selected 9998 images out of 9998 total.


In [8]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV

# Convert image arrays into a feature matrix
X = np.stack(df['image_array'].values)

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])  # Convert categorical labels to numbers

# Train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n✅ Data prepared successfully!")
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")



✅ Data prepared successfully!
Training samples: 7998, Testing samples: 2000
